In [1]:
import pandas as pd
import glob
import os

In [2]:
raw_path = "../data/raw/"
files = glob.glob(os.path.join(raw_path, "*.csv"))
files

['../data/raw\\ASRS_DBOnline_012020-062020.csv',
 '../data/raw\\ASRS_DBOnline_062020-122020.csv',
 '../data/raw\\ASRS_DBOnline_062021-122021.csv',
 '../data/raw\\ASRS_DBOnline_062022-122022.csv',
 '../data/raw\\ASRS_DBOnline_122020-062021.csv',
 '../data/raw\\ASRS_DBOnline_122021-062022.csv']

In [ ]:
with open(files[0], 'r', encoding='utf-8', errors='ignore') as f:
    for i in range(10):
        print(i, f.readline())

In [23]:
test = pd.read_csv(files[0], on_bad_lines='skip', engine='python', header=1)
test.columns.tolist()

['ACN',
 'Date',
 'Local Time Of Day',
 'Locale Reference',
 'State Reference',
 'Relative Position.Angle.Radial',
 'Relative Position.Distance.Nautical Miles',
 'Altitude.AGL.Single Value',
 'Altitude.MSL.Single Value',
 'Latitude / Longitude (UAS)',
 'Flight Conditions',
 'Weather Elements / Visibility',
 'Work Environment Factor',
 'Light',
 'Ceiling',
 'RVR.Single Value',
 'ATC / Advisory',
 'Aircraft Operator',
 'Make Model Name',
 'Aircraft Zone',
 'Crew Size',
 'Operating Under FAR Part',
 'Flight Plan',
 'Mission',
 'Nav In Use',
 'Flight Phase',
 'Route In Use',
 'Airspace',
 'Maintenance Status.Maintenance Deferred',
 'Maintenance Status.Records Complete',
 'Maintenance Status.Released For Service',
 'Maintenance Status.Required / Correct Doc On Board',
 'Maintenance Status.Maintenance Type',
 'Maintenance Status.Maintenance Items Involved',
 'Cabin Lighting',
 'Number Of Seats.Number',
 'Passengers On Board.Number',
 'Crew Size Flight Attendant.Number Of Crew',
 'Airspace Au

In [24]:
def count_skipped_rows(file):
    # Count raw lines minus header rows (we skip 2 header rows)
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        total_lines = sum(1 for _ in f) - 2

    # Load using correct header row
    df = pd.read_csv(file, on_bad_lines='skip', engine='python', header=1)

    loaded_rows = len(df)
    skipped = total_lines - loaded_rows
    return skipped, loaded_rows, total_lines

for f in files:
    skipped, loaded, total = count_skipped_rows(f)
    print(f"\nFile: {os.path.basename(f)}")
    print(f"Total lines (minus header rows): {total}")
    print(f"Loaded rows: {loaded}")
    print(f"Skipped rows: {skipped}")



File: ASRS_DBOnline_012020-062020.csv
Total lines (minus header rows): 1957
Loaded rows: 1946
Skipped rows: 11

File: ASRS_DBOnline_062020-122020.csv
Total lines (minus header rows): 3454
Loaded rows: 3429
Skipped rows: 25

File: ASRS_DBOnline_062021-122021.csv
Total lines (minus header rows): 2631
Loaded rows: 2151
Skipped rows: 480

File: ASRS_DBOnline_062022-122022.csv
Total lines (minus header rows): 3083
Loaded rows: 3031
Skipped rows: 52

File: ASRS_DBOnline_122020-062021.csv
Total lines (minus header rows): 2808
Loaded rows: 2763
Skipped rows: 45

File: ASRS_DBOnline_122021-062022.csv
Total lines (minus header rows): 3289
Loaded rows: 3215
Skipped rows: 74


In [25]:
cleaned_dfs = []

for f in files:
    print("Loading:", os.path.basename(f))

    # Use the REAL header row (row 1)
    temp = pd.read_csv(f, on_bad_lines='skip', engine='python', header=1)

    # Drop junk columns
    temp = temp.drop(columns=[' ', 'Unnamed: 125'], errors='ignore')

    # Clean column names
    temp.columns = temp.columns.str.strip()

    # Add metadata
    temp['source_file'] = os.path.basename(f)

    cleaned_dfs.append(temp)

merged = pd.concat(cleaned_dfs, ignore_index=True)

print("\nMerged shape:", merged.shape)
merged.head()

Loading: ASRS_DBOnline_012020-062020.csv
Loading: ASRS_DBOnline_062020-122020.csv
Loading: ASRS_DBOnline_062021-122021.csv
Loading: ASRS_DBOnline_062022-122022.csv
Loading: ASRS_DBOnline_122020-062021.csv
Loading: ASRS_DBOnline_122021-062022.csv

Merged shape: (16535, 126)


,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial,Relative Position.Distance.Nautical Miles,Altitude.AGL.Single Value,Altitude.MSL.Single Value,Latitude / Longitude (UAS),...,When Detected,Result,Contributing Factors / Situations,Primary Problem,Narrative,Callback,Narrative.1,Callback.1,Synopsis,source_file
0,1714400,202001.0,0001-0600,ORD.Airport,IL,NaN,NaN,NaN,15000.0,NaN,...,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft; Company Policy,Company Policy,To be clear I did not have a lot going on and ...,NaN,NaN,NaN,A TRACON Departure Controller reported two air...,ASRS_DBOnline_012020-062020.csv
1,1714553,202001.0,1801-2400,ZHU.ARTCC,TX,NaN,NaN,NaN,14000.0,NaN,...,In-flight,Flight Crew Became Reoriented; General Physica...,Human Factors; Environment - Non Weather Related,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...,NaN,NaN,NaN,CRJ Captain reported that a laser shone repeat...,ASRS_DBOnline_012020-062020.csv
2,1714675,202001.0,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,9000.0,NaN,...,In-flight,Flight Crew Returned To Departure Airport; Fli...,Aircraft; Weather; Procedure,Weather,I was boarding the aircraft when a my destinat...,NaN,NaN,NaN,Pilot reported weather and a closed destinatio...,ASRS_DBOnline_012020-062020.csv
3,1714708,202001.0,0601-1200,ZZZ.Airport,US,NaN,NaN,0.0,NaN,NaN,...,In-flight,General None Reported / Taken,Procedure; Logbook Entry; Human Factors,Ambiguous,I had a ferry flight that went out of service ...,NaN,We departed ZZZ ferrying an un-airworthy aircr...,NaN,Air carrier Dispatcher and flight crew reporte...,ASRS_DBOnline_012020-062020.csv
4,1714843,202001.0,1801-2400,OSU.Airport,OH,NaN,0.0,0.0,NaN,NaN,...,In-flight,Flight Crew Diverted; Flight Crew Landed in Em...,Aircraft; Airport,Aircraft,We were flying at 2500 ft. I started losing el...,NaN,NaN,NaN,A Pilot reported a loss of electrical power at...,ASRS_DBOnline_012020-062020.csv


In [26]:
interim_path = "../data/interim/"
os.makedirs(interim_path, exist_ok=True)

output_file = os.path.join(interim_path, "asrs_merged_3yrs.csv")
merged.to_csv(output_file, index=False)

print("Saved merged dataset to:", output_file)
print("Final shape:", merged.shape)

Saved merged dataset to: ../data/interim/asrs_merged_3yrs.csv
Final shape: (16535, 126)
